# HUC Inference — Wetland Prevalence (data-heavy companion)

Split out of `dl_10_factorial_viz.ipynb` because it reads the large per-HUC
prediction **GeoTIFFs** in `Data/HUC_DL_Predictions/` (multi-GB; gitignored).
The lightweight `dl_10_factorial_viz.ipynb` covers §1–§7 from small CSV/JSON
that *do* sync through git.

**Before running:** rsync the HUC rasters to `Data/HUC_DL_Predictions/`
(`DLpred_*_huc_*.tif`) — they are not in the repo. Set `EXP_VERSION` in the env
to target a repeat run (e.g. `HUC_DL_Predictions_v2`).


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# seaborn drives every heatmap / confusion matrix. Self-heal if the active
# kernel doesn't have it (this repo has several Python envs floating around).
try:
    import seaborn as sns
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "seaborn"])
    import seaborn as sns
sns.set_style("whitegrid")

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "axes.grid": True, "grid.alpha": 0.3,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
})

# --- Locate the results tree -------------------------------------------------
# Notebook lives in Python_Code_Analysis/DL_Pipeline_v2/. Walk up to the repo
# root and find Models/factorial_results so this works no matter the cwd.
# --- Experiment version selector ---------------------------------------------
# Base factorial -> "" (dirs: Models/factorial_results, results_patchcurve, ...).
# A repeat run (EXECUTION.md §11) suffixes every results root: EXP_VERSION="v2"
# -> Models/factorial_results_v2, results_patchcurve_v2, HUC_DL_Predictions_v2.
# Read from the env so the notebook can target a version without editing code,
# or hard-set the literal here.
EXP_VERSION = os.environ.get("EXP_VERSION", "")     # "" = base; "v2", "v3", ...
_SUFFIX = f"_{EXP_VERSION}" if EXP_VERSION else ""

def find_results_dir() -> Path:
    here = Path.cwd()
    for base in [here, *here.parents]:
        cand = base / "Models" / f"factorial_results{_SUFFIX}"
        if cand.is_dir():
            return cand
    # fall back to the canonical absolute path
    cand = Path(f"/ibstorage/anthony/NYS_Wetlands_DL/Models/factorial_results{_SUFFIX}")
    if cand.is_dir():
        return cand
    raise FileNotFoundError(f"Could not locate Models/factorial_results{_SUFFIX}")

RESULTS_DIR = find_results_dir()
ANALYSIS_DIR = RESULTS_DIR / "analysis"
FIG_DIR = ANALYSIS_DIR / "figures"      # PNGs are saved here for the write-up
FIG_DIR.mkdir(exist_ok=True, parents=True)
print("results :", RESULTS_DIR)
print("analysis:", ANALYSIS_DIR)
print("figures :", FIG_DIR)

# --- Canonical ordering / labels (mirrors dl_experiment_config.py) -----------
CONFIG_ORDER = [
    "fld_nolidar_leafon", "fld_nolidar_leafoff",
    "fld_chm_leafon",     "fld_chm_leafoff",
    "fld_chmret_leafon",  "fld_chmret_leafoff",
    "nwi_chmret_leafoff", "flddeg_chmret_leafoff",
]
CLASS_ORDER = ["EMW", "FSW", "SSW", "UPL"]
CLASS_FULL = {"EMW": "Emergent", "FSW": "Forested", "SSW": "Scrub-Shrub", "UPL": "Upland"}

# A stable color per config and per class (used across every figure).
_cfg_cmap = plt.get_cmap("tab10")
CONFIG_COLORS = {c: _cfg_cmap(i % 10) for i, c in enumerate(CONFIG_ORDER)}
CLASS_COLORS = {"EMW": "#2ca02c", "FSW": "#1f77b4", "SSW": "#ff7f0e", "UPL": "#8c564b"}

def order_configs(seq):
    """Keep CONFIG_ORDER, drop configs absent from `seq`, append any extras."""
    present = [c for c in CONFIG_ORDER if c in set(seq)]
    extra = [c for c in seq if c not in CONFIG_ORDER]
    return present + sorted(set(extra))

def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")


In [ ]:
# Locate the HUC prediction rasters (sibling of Models/ under the repo root).
import re
MODELS_DIR = RESULTS_DIR.parent
PRED_DIR   = MODELS_DIR.parent / "Data" / f"HUC_DL_Predictions{_SUFFIX}"
print(("ok      " if PRED_DIR.exists() else "MISSING "), PRED_DIR)


## 8 · HUC inference — wetland prevalence

Per-HUC class composition and overall wetland prevalence (EMW+FSW+SSW) from the
sliding-window predictions in `Data/HUC_DL_Predictions/`. Area uses the raster
pixel size (1 m² → ha). A map-scale sanity check that predicted wetland fraction
lands near the expected landscape value (plan §9, `run_predict_factorial.sh`).

In [ ]:
# Per-class area & wetland prevalence from the HUC class GeoTIFFs (nodata=255).
import rasterio
class_tifs = (sorted(p for p in PRED_DIR.glob("DLpred_*_huc_*.tif")
                     if not p.name.endswith("_probs.tif"))
              if PRED_DIR.is_dir() else [])
if not class_tifs:
    print("No HUC class predictions under", PRED_DIR, "- skipping section 8.")
else:
    rows = []
    for tif in class_tifs:
        mh = re.search(r"cluster_(\d+)_huc_(\d+)", tif.name)
        huc = mh.group(2) if mh else tif.stem
        with rasterio.open(tif) as ds:
            px_ha = abs(ds.res[0] * ds.res[1]) / 10_000.0     # m^2 -> ha (CRS in metres)
            counts = np.zeros(256, dtype=np.int64)
            for _, win in ds.block_windows(1):                # windowed: full raster too big to hold
                counts += np.bincount(ds.read(1, window=win).ravel(), minlength=256)
        per = {CLASS_ORDER[c]: int(counts[c]) for c in range(len(CLASS_ORDER))}
        tot = sum(per.values())
        row = {"huc": huc, "valid_px": tot}
        for c in CLASS_ORDER:
            row[f"{c}_ha"]  = per[c] * px_ha
            row[f"{c}_pct"] = 100.0 * per[c] / tot if tot else np.nan
        row["wetland_pct"] = sum(row[f"{c}_pct"] for c in ["EMW", "FSW", "SSW"])
        rows.append(row)
    prev = pd.DataFrame(rows).sort_values("huc").reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bottom = np.zeros(len(prev))
    for c in CLASS_ORDER:
        axes[0].bar(prev["huc"], prev[f"{c}_pct"], bottom=bottom, label=c,
                    color=CLASS_COLORS.get(c), edgecolor="black", linewidth=0.3)
        bottom += prev[f"{c}_pct"].values
    axes[0].set_ylabel("% of valid pixels"); axes[0].set_title("Predicted class composition per HUC")
    axes[0].tick_params(axis="x", rotation=45); axes[0].legend(ncol=4, fontsize=8)
    axes[1].bar(prev["huc"], prev["wetland_pct"], color="#117733", edgecolor="black", linewidth=0.4)
    for i, v in enumerate(prev["wetland_pct"]):
        axes[1].text(i, v + 0.3, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
    axes[1].set_ylabel("wetland %  (EMW+FSW+SSW)"); axes[1].set_title("Predicted wetland prevalence per HUC")
    axes[1].tick_params(axis="x", rotation=45)
    fig.tight_layout(); savefig(fig, "08_huc_class_prevalence"); plt.show()
    prev.round(2)